In [1]:
# Last update on 30/6/2026

# I'll search mRNAs with target sites of selected miRNAs.
# miR-494 in miRBase = Mir-154-P18 in MirGeneDB v2.1 = Mir-154-P9 in MirGeneDB v3.0.


In [2]:
import pandas as pd
from Bio import Entrez
from Bio.Seq import Seq
import re
import numpy as np

In [10]:
# We are looking for 599-specific downregulated mRNAs

#599-Mir-154-P18_3p_7merSupp (=miR-494_7merSupp)
sSeq599 = 'tgaaacattgtggggattg'.upper()
#600-Mir-154-P18_3p_7mer	 (=miR-494_7mer)
sSeq600 = 'tgaaacattgtgccctttg'.upper()

In [5]:
sMrnaFile = '/home/kwonschul/data/genome/salmon_hg38_canonical_250114/Canonical_hg38_260630_mrna.txt'
sUtr3File = '/home/kwonschul/data/genome/salmon_hg38_canonical_250114/Canonical_hg38_260630_utr3.txt'
dfMrna = pd.read_table(sMrnaFile)
dfUtr3 = pd.read_table(sUtr3File)
dfMrna

,id,mrna
0,A1BG|ENSG00000121410.12|ENST00000263100.8,ATTGCTGCAGACGCTCACCCCAGACACTCACTGCACCGGAGTGAGC...
1,A1CF|ENSG00000148584.16|ENST00000373997.8,ATAATCAAGGAAACCTTTTCCGGGTGGGGATCTCTGAAATTACTCA...
2,A2M|ENSG00000175899.15|ENST00000318602.12,GGGACCAGATGGATTGTAGGGAGTAGGGTACAATACAGTCTGTTCT...
3,A2ML1|ENSG00000166535.20|ENST00000299698.12,GACCCTGGAAAAATCTGTCTCACCCACAAAGATGTGGGCTCAGCTC...
4,A4GALT|ENSG00000128274.17|ENST00000642412.2,ACTGCCCTCCGGCCGCCGCGCCGCCCGCCCGCCGGGGCCCCGCTGT...
...,...,...
19486,ZYG11A|ENSG00000203995.11|ENST00000371528.2,GCTAGCTCCGTGTGCCTCGCAGGCGTGGTGGGCGCGTCCTGGCAGC...
19487,ZYG11B|ENSG00000162378.13|ENST00000294353.7,GGAGTCTGCGCTCTGGTTCGGGCTGCGGCTGCGGCTGCGGCTGCGG...
19488,ZYX|ENSG00000159840.16|ENST00000322764.10,GCAGAGTCTGCGGACCCGGCGCCGAGGCGGCCACCCGAGACGCGGC...
19489,ZZEF1|ENSG00000074755.15|ENST00000381638.7,AGGAAGCCGGAAGCCGCAGGGGCCGCCGTCGTCTCCTCCGCGTCCC...


In [7]:
dfChosen = dfMrna.copy()

In [8]:
# scan 3' suppl sites

In [9]:
def get_seed_type_target(sMirna):
    sMirna = Seq(sMirna.upper())
    # 5'-3' direction, target mRNA sequence
    s8mer_Target = str(sMirna[1:8].reverse_complement()) + 'A' 
    s7merm8_Target = str(sMirna[1:8].reverse_complement())
    s7merA1_Target = str(sMirna[1:7].reverse_complement()) + 'A'     
    s6mer_Target = str(sMirna[1:7].reverse_complement())
    s1316_Target = str(sMirna[12:16].reverse_complement())
    s1215_Target = str(sMirna[11:15].reverse_complement())
    s1417_Target = str(sMirna[13:17].reverse_complement())        

    return (s8mer_Target,
            s7merm8_Target,
            s7merA1_Target,
            s6mer_Target,
            s1316_Target,
            s1215_Target,
            s1417_Target)            

def contains_seed(sTargetSeq, sSeed_Target):
    sTargetSeq = sTargetSeq.upper()
    match = re.findall(sSeed_Target, sTargetSeq)
    if len(match) == 0:
        sReturn = '.'
    else:
        sReturn = ','.join(match)
    return sReturn
    
def contains_suppl_region(sTargetSeq, s6mer_Target, sSuppl_Target, nMinDist=1, nMaxDist=20):
    sTargetSeq = sTargetSeq.upper()
    # Define the regular expression
    pattern = r"({0}.{{{1},{2}}}{3})".format(sSuppl_Target, nMinDist, nMaxDist, s6mer_Target)
    
    # Find a match in the string (list will be returned)
    match = re.findall(pattern, sTargetSeq)

    # If a match is found, return the value; otherwise, return the empty
    if len(match) == 0:
        sReturn = '.'
    else:
        sReturn = ','.join(match)
    return sReturn

In [11]:
lMirnaName = ['miR-494-7merSupp', 'miR-494-7mer']
lMirnaSeq = [sSeq599, sSeq600]

for sMirnaName, sMirnaSeq in zip(lMirnaName, lMirnaSeq):
    print(sMirnaName, sMirnaSeq)    

    dfChosen['{}.8mer_Target'.format(sMirnaName)],\
    dfChosen['{}.7merm8_Target'.format(sMirnaName)],\
    dfChosen['{}.7merA1_Target'.format(sMirnaName)],\
    dfChosen['{}.6mer_Target'.format(sMirnaName)],\
    dfChosen['{}.1316_Target'.format(sMirnaName)],\
    dfChosen['{}.1215_Target'.format(sMirnaName)],\
    dfChosen['{}.1417_Target'.format(sMirnaName)] = zip(*dfChosen.apply(lambda row: get_seed_type_target(sMirnaSeq), axis=1))
  
    dfChosen['{}.contains_8mer'.format(sMirnaName)] = dfChosen.apply(lambda x: contains_seed(x['mrna'], 
                                                                               x['{}.8mer_Target'.format(sMirnaName)]), 
                                                   axis=1)
    dfChosen['{}.contains_7merm8'.format(sMirnaName)] = dfChosen.apply(lambda x: contains_seed(x['mrna'], 
                                                                                 x['{}.7merm8_Target'.format(sMirnaName)]), 
                                                   axis=1)
    dfChosen['{}.contains_7merA1'.format(sMirnaName)] = dfChosen.apply(lambda x: contains_seed(x['mrna'], 
                                                                                 x['{}.7merA1_Target'.format(sMirnaName)]), 
                                                   axis=1)
    dfChosen['{}.contains_6mer_1316'.format(sMirnaName)] = dfChosen.apply(lambda x: contains_suppl_region(x['mrna'], 
                                                                                    x['{}.6mer_Target'.format(sMirnaName)],
                                                                                    x['{}.1316_Target'.format(sMirnaName)]), 
                                                   axis=1)
    dfChosen['{}.contains_6mer_1215'.format(sMirnaName)] = dfChosen.apply(lambda x: contains_suppl_region(x['mrna'], 
                                                                                    x['{}.6mer_Target'.format(sMirnaName)],
                                                                                    x['{}.1215_Target'.format(sMirnaName)]), 
                                                   axis=1)
    dfChosen['{}.contains_6mer_1417'.format(sMirnaName)] = dfChosen.apply(lambda x: contains_suppl_region(x['mrna'], 
                                                                                    x['{}.6mer_Target'.format(sMirnaName)],
                                                                                    x['{}.1417_Target'.format(sMirnaName)]), 
                                                   axis=1)
    

dfChosen

miR-494-7merSupp TGAAACATTGTGGGGATTG
miR-494-7mer TGAAACATTGTGCCCTTTG


,id,mrna,miR-494-7merSupp.8mer_Target,miR-494-7merSupp.7merm8_Target,miR-494-7merSupp.7merA1_Target,miR-494-7merSupp.6mer_Target,miR-494-7merSupp.1316_Target,miR-494-7merSupp.1215_Target,miR-494-7merSupp.1417_Target,miR-494-7merSupp.contains_8mer,...,miR-494-7mer.6mer_Target,miR-494-7mer.1316_Target,miR-494-7mer.1215_Target,miR-494-7mer.1417_Target,miR-494-7mer.contains_8mer,miR-494-7mer.contains_7merm8,miR-494-7mer.contains_7merA1,miR-494-7mer.contains_6mer_1316,miR-494-7mer.contains_6mer_1215,miR-494-7mer.contains_6mer_1417
0,A1BG|ENSG00000121410.12|ENST00000263100.8,ATTGCTGCAGACGCTCACCCCAGACACTCACTGCACCGGAGTGAGC...,ATGTTTCA,ATGTTTC,TGTTTCA,TGTTTC,TCCC,CCCC,ATCC,.,...,TGTTTC,AGGG,GGGC,AAGG,.,.,.,.,.,.
1,A1CF|ENSG00000148584.16|ENST00000373997.8,ATAATCAAGGAAACCTTTTCCGGGTGGGGATCTCTGAAATTACTCA...,ATGTTTCA,ATGTTTC,TGTTTCA,TGTTTC,TCCC,CCCC,ATCC,.,...,TGTTTC,AGGG,GGGC,AAGG,.,"ATGTTTC,ATGTTTC","TGTTTCA,TGTTTCA",.,.,AAGGAACCTGTTTC
2,A2M|ENSG00000175899.15|ENST00000318602.12,GGGACCAGATGGATTGTAGGGAGTAGGGTACAATACAGTCTGTTCT...,ATGTTTCA,ATGTTTC,TGTTTCA,TGTTTC,TCCC,CCCC,ATCC,.,...,TGTTTC,AGGG,GGGC,AAGG,.,.,TGTTTCA,.,.,AAGGACAATGGCTGTTTC
3,A2ML1|ENSG00000166535.20|ENST00000299698.12,GACCCTGGAAAAATCTGTCTCACCCACAAAGATGTGGGCTCAGCTC...,ATGTTTCA,ATGTTTC,TGTTTCA,TGTTTC,TCCC,CCCC,ATCC,.,...,TGTTTC,AGGG,GGGC,AAGG,.,"ATGTTTC,ATGTTTC",.,.,.,.
4,A4GALT|ENSG00000128274.17|ENST00000642412.2,ACTGCCCTCCGGCCGCCGCGCCGCCCGCCCGCCGGGGCCCCGCTGT...,ATGTTTCA,ATGTTTC,TGTTTCA,TGTTTC,TCCC,CCCC,ATCC,.,...,TGTTTC,AGGG,GGGC,AAGG,.,.,.,.,.,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19486,ZYG11A|ENSG00000203995.11|ENST00000371528.2,GCTAGCTCCGTGTGCCTCGCAGGCGTGGTGGGCGCGTCCTGGCAGC...,ATGTTTCA,ATGTTTC,TGTTTCA,TGTTTC,TCCC,CCCC,ATCC,.,...,TGTTTC,AGGG,GGGC,AAGG,.,"ATGTTTC,ATGTTTC",.,.,.,.
19487,ZYG11B|ENSG00000162378.13|ENST00000294353.7,GGAGTCTGCGCTCTGGTTCGGGCTGCGGCTGCGGCTGCGGCTGCGG...,ATGTTTCA,ATGTTTC,TGTTTCA,TGTTTC,TCCC,CCCC,ATCC,.,...,TGTTTC,AGGG,GGGC,AAGG,.,ATGTTTC,"TGTTTCA,TGTTTCA",.,.,.
19488,ZYX|ENSG00000159840.16|ENST00000322764.10,GCAGAGTCTGCGGACCCGGCGCCGAGGCGGCCACCCGAGACGCGGC...,ATGTTTCA,ATGTTTC,TGTTTCA,TGTTTC,TCCC,CCCC,ATCC,.,...,TGTTTC,AGGG,GGGC,AAGG,.,.,.,.,.,.
19489,ZZEF1|ENSG00000074755.15|ENST00000381638.7,AGGAAGCCGGAAGCCGCAGGGGCCGCCGTCGTCTCCTCCGCGTCCC...,ATGTTTCA,ATGTTTC,TGTTTCA,TGTTTC,TCCC,CCCC,ATCC,ATGTTTCA,...,TGTTTC,AGGG,GGGC,AAGG,ATGTTTCA,"ATGTTTC,ATGTTTC",TGTTTCA,.,.,.


In [12]:
sOutputFile = '/home/kwonschul/data/hku_NGS/43_novaseq50_250107/analysis-260630/nonredun_mir494_seedsuppl_in_mRNA_260630.csv'
dfChosen.to_csv(sOutputFile, index=False)